# Lab 07 Challenge Solution: Smart Study Buddy — Mini Agent

## Setup: Imports and LLM

In [1]:
import json
import math
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOllama(model="llama3.2:1b")

## Part A: Tools

The starter tools are provided. The `explain_code` tool (TODO 3) is added later.

In [2]:
def calculator(expression: str) -> str:
    """Calculate a math expression."""
    try:
        allowed = {"__builtins__": {}, "math": math}
        return str(eval(expression, allowed))
    except Exception as e:
        return f"Error: {e}"


def define_word(word: str) -> str:
    """Look up the definition of a word."""
    definitions = {
        "algorithm": "A step-by-step procedure for solving a problem or accomplishing a task.",
        "api": "Application Programming Interface \u2014 a way for software programs to communicate with each other.",
        "variable": "A named storage location in a program that holds a value which can change.",
        "function": "A reusable block of code that performs a specific task.",
        "loop": "A programming construct that repeats a block of code multiple times.",
        "recursion": "When a function calls itself to solve smaller instances of the same problem.",
        "testing": "The practice of verifying that software behaves as expected through automated or manual checks.",
        "database": "An organized collection of structured data stored electronically for efficient retrieval and management.",
    }
    return definitions.get(word.lower().strip(), f"Definition not found for: {word}")


def generate_quiz(topic: str) -> str:
    """Generate a quick quiz question about a topic."""
    quizzes = {
        "python": "Q: What keyword is used to define a function in Python?\nA) func  B) def  C) function  D) define\nCorrect: B) def",
        "git": "Q: What command creates a new Git branch?\nA) git new  B) git create  C) git branch  D) git fork\nCorrect: C) git branch",
        "testing": "Q: Which testing framework is commonly used in Python?\nA) JUnit  B) pytest  C) Mocha  D) RSpec\nCorrect: B) pytest",
        "linux": "Q: What command lists files in a directory?\nA) dir  B) show  C) ls  D) list\nCorrect: C) ls",
    }
    for key, quiz in quizzes.items():
        if key in topic.lower():
            return quiz
    return f"No quiz available for: {topic}. Try: python, git, testing, linux."

In [3]:
TOOLS = {
    "calculator": {"fn": calculator, "desc": "Calculate a math expression (e.g., '17 * 28', 'math.sqrt(144)')"},
    "define_word": {"fn": define_word, "desc": "Look up the definition of a programming/tech term"},
    "generate_quiz": {"fn": generate_quiz, "desc": "Generate a quiz question about a topic (python, git, testing, linux)"},
}

## Part B: System Prompt (TODO 1 Solution)

The prompt is built dynamically from the TOOLS dict so adding new tools automatically updates it.

In [4]:
# TODO 1 Solution: Build SYSTEM_PROMPT dynamically from TOOLS
tool_lines = "\n".join(f"- {name}(argument): {info['desc']}" for name, info in TOOLS.items())

SYSTEM_PROMPT = f"""You are a Smart Study Buddy that helps students learn programming concepts.
You are friendly, encouraging, and patient.

Available tools:
{tool_lines}

When you need a tool, respond with EXACTLY this JSON:
{{"tool": "tool_name", "argument": "the argument"}}

If you can answer without a tool, respond directly.
For complex questions, think step by step before answering.
Remember what the student has asked before and build on it.
Be encouraging — celebrate when they get things right!"""

## Part C: Agent Loop (TODO 2 Solution)

The ReAct loop with memory, tool parsing, and up to 3 tool calls per turn.

In [5]:
# TODO 2 Solution: Full agent loop implementation
def run_agent(user_message: str, history: list) -> tuple[str, list]:
    """Run one turn of the agent with ReAct loop."""
    # Step 1: Add user message to history
    history.append(HumanMessage(content=user_message))

    # Step 2: ReAct loop (max 3 tool calls per turn)
    response_text = ""
    for _ in range(3):
        response = llm.invoke(history)
        response_text = response.content

        # Try to parse a tool call
        try:
            start = response_text.index("{")
            end = response_text.rindex("}") + 1
            call = json.loads(response_text[start:end])
            tool_name = call.get("tool", "")
            argument = call.get("argument", "")

            if tool_name in TOOLS:
                result = TOOLS[tool_name]["fn"](argument) if argument else TOOLS[tool_name]["fn"]()
                print(f"  [Tool: {tool_name}('{argument}') -> {result}]")
                # Add tool interaction to history
                history.append(AIMessage(content=response_text))
                history.append(HumanMessage(content=f"Tool result: {result}\nNow respond to the student based on this result. Be friendly and helpful."))
                continue  # Let the LLM process the result
            else:
                # Unknown tool — treat as direct answer
                break
        except (json.JSONDecodeError, ValueError):
            # No tool call — it's a direct answer
            break

    # Step 3: Add final response to history
    history.append(AIMessage(content=response_text))
    # Step 4: Return
    return response_text, history

## Part D: Test and Extend (TODO 3 Solution)

Test the agent, then add the `explain_code` tool and rebuild the prompt.

In [ ]:
# Test the agent
history = [SystemMessage(content=SYSTEM_PROMPT)]

test_messages = [
    "Hi! I'm learning programming. Can you help me?",
    "What does 'algorithm' mean?",
    "What is 17 multiplied by 2847?",                            # Forces calculator use
    "Give me a quiz about Python!",
    "What were the two topics I asked you to look up or calculate?",  # Tests memory
]

for msg in test_messages:
    print(f"\nYou: {msg}")
    reply, history = run_agent(msg, history)
    print(f"Bot: {reply}")
    print(f"     [Memory: {len(history)} messages]")

In [ ]:
# TODO 3 Solution: Add example_code tool, rebuild prompt, and test

def example_code(topic: str) -> str:
    """Return a short, runnable code snippet for a programming concept."""
    examples = {
        "function": "def greet(name):\n    return f'Hello, {name}!'\n\nprint(greet('World'))  # Output: Hello, World!",
        "loop": "for i in range(5):\n    print(i)  # Prints 0, 1, 2, 3, 4",
        "list": "fruits = ['apple', 'banana', 'cherry']\nfruits.append('date')\nprint(fruits[0])  # apple",
        "dictionary": "person = {'name': 'Alice', 'age': 30}\nprint(person['name'])  # Alice",
        "class": "class Dog:\n    def __init__(self, name):\n        self.name = name\n    def bark(self):\n        return f'{self.name} says Woof!'\n\ndog = Dog('Rex')\nprint(dog.bark())",
        "exception": "try:\n    x = 1 / 0\nexcept ZeroDivisionError:\n    print('Cannot divide by zero!')",
    }
    for key, code in examples.items():
        if key in topic.lower():
            return f"Example for '{key}':\n{code}"
    return f"No example for '{topic}'. Try: function, loop, list, dictionary, class, exception."

# Add to TOOLS dict
TOOLS["example_code"] = {"fn": example_code, "desc": "Get a short, runnable code example for a programming concept (function, loop, list, dictionary, class, exception)"}

# Rebuild SYSTEM_PROMPT dynamically (same logic as TODO 1, now includes example_code)
tool_lines = "\n".join(f"- {name}(argument): {info['desc']}" for name, info in TOOLS.items())

SYSTEM_PROMPT = f"""You are a Smart Study Buddy that helps students learn programming concepts.
You are friendly, encouraging, and patient.

Available tools:
{tool_lines}

When you need a tool, respond with EXACTLY this JSON:
{{"tool": "tool_name", "argument": "the argument"}}

If you can answer without a tool, respond directly.
For complex questions, think step by step before answering.
Remember what the student has asked before and build on it.
Be encouraging — celebrate when they get things right!"""

# Test the new tool
print("Testing example_code tool:")
history2 = [SystemMessage(content=SYSTEM_PROMPT)]
reply, history2 = run_agent("Can you show me a code example of a loop?", history2)
print(f"Bot: {reply}")

In [ ]:
# Validation
score = 0
checks = []

# Check TODO 1: SYSTEM_PROMPT
if SYSTEM_PROMPT == "___":
    checks.append(("System prompt built", "TODO"))
    checks.append(("Prompt mentions tools", "TODO"))
    checks.append(("Prompt has JSON format", "TODO"))
else:
    if len(SYSTEM_PROMPT) > 50:
        checks.append(("System prompt built (length > 50)", "PASS"))
        score += 1
    else:
        checks.append(("System prompt built (too short)", "FAIL"))
    if "calculator" in SYSTEM_PROMPT and "define_word" in SYSTEM_PROMPT:
        checks.append(("Prompt mentions tools", "PASS"))
        score += 1
    else:
        checks.append(("Prompt mentions tools", "FAIL"))
    if '{"tool"' in SYSTEM_PROMPT or '"tool"' in SYSTEM_PROMPT:
        checks.append(("Prompt has JSON format instruction", "PASS"))
        score += 1
    else:
        checks.append(("Prompt has JSON format instruction", "FAIL"))

# Check TODO 2: run_agent — basic response + tool invocation
test_history = [SystemMessage(content=SYSTEM_PROMPT if SYSTEM_PROMPT != "___" else "test")]
result, _ = run_agent("Hello", test_history)
if result == "___":
    checks.append(("run_agent returns response", "TODO"))
    checks.append(("run_agent invokes tools", "TODO"))
else:
    if isinstance(result, str) and len(result) > 0:
        checks.append(("run_agent returns response", "PASS"))
        score += 1
    else:
        checks.append(("run_agent returns response", "FAIL"))
    # Verify tool invocation: a math query should grow history beyond 3 messages
    # (SystemMessage + HumanMessage + AIMessage(tool_call) + HumanMessage(result) + AIMessage(final) = 5)
    tool_history = [SystemMessage(content=SYSTEM_PROMPT if SYSTEM_PROMPT != "___" else "test")]
    _, tool_history = run_agent("What is 17 multiplied by 2847?", tool_history)
    if len(tool_history) >= 5:
        checks.append(("run_agent invokes tools (history grew)", "PASS"))
        score += 1
    else:
        checks.append(("run_agent invokes tools (answered without tool call)", "FAIL"))

# Check TODO 3: example_code tool
if "example_code" in TOOLS:
    checks.append(("example_code tool added", "PASS"))
    score += 1
    r = TOOLS["example_code"]["fn"]("loop")
    if isinstance(r, str) and "\n" in r:
        checks.append(("example_code returns a code snippet", "PASS"))
        score += 1
    else:
        checks.append(("example_code returns a code snippet (no code found)", "FAIL"))
else:
    checks.append(("example_code tool added", "TODO"))
    checks.append(("example_code returns a code snippet", "TODO"))

for check, status in checks:
    print(f"[{status}] {check}")
print(f"\nScore: {score}/7")

## Part E (Bonus): Interactive Mode

Uncomment the code below to chat with the Study Buddy interactively.
Type `quit` to exit.

In [ ]:
# Uncomment to run interactively:

# history = [SystemMessage(content=SYSTEM_PROMPT)]
# while True:
#     user_input = input("\nYou: ").strip()
#     if user_input.lower() == "quit":
#         print("Goodbye! Keep learning — you're doing great!")
#         break
#     reply, history = run_agent(user_input, history)
#     print(f"Bot: {reply}")

## Part F (Bonus): Add Even More Tools

The solution added `example_code` as TODO 3. More ideas:
- `compare(a_and_b)`: Compare two technologies (e.g., "Python vs JavaScript")
- `acronym(letters)`: Expand a tech acronym (API, REST, SQL)
- `debug_hint(error)`: Return a tip for a common error type (NameError, IndexError, TypeError)

Add them to `TOOLS` and rebuild `SYSTEM_PROMPT` (the dynamic builder handles it automatically!).

## Takeaways

- The agent combines **ReAct reasoning**, **tool use**, and **conversational memory** in a single loop
- Dynamic prompt building means adding tools is effortless — just add to TOOLS and rebuild
- Memory lets the bot reference earlier messages (e.g., "What two topics did I ask about?")
- The ReAct loop decides whether to call a tool or answer directly
- All 3 TODOs should score 7/7 in the validation cell